# CS781 Phase 3 — Notebook P3_01: Rhetorical-role data, statistics, tokenisation, baselines

Nothing is trained with a neural network here. The notebook makes sure we understand the data and the metric
before modelling, and prepares everything P3_02 needs.

| Step | What happens |
|---|---|
| 1 | download `Phase_3_RR` from Hugging Face, detect the file format, parse documents (sentences + labels) |
| 2 | checks: one label per sentence, labels in the vocabulary, no ID collisions between CL and IT, empty sentences |
| 3 | statistics per domain: label distribution (imbalance), sentence and document lengths |
| 4 | **structure**: where in a document each role appears, how "blocky" labels are (runs), most common transitions |
| 5 | the official metric (copied from `evaluate_phase3.py`) and what it implies (macro over all 13 labels) |
| 6 | tokenisation with the Phase-1 tokenizer (same normalisation as pretraining) |
| 7 | baselines on dev: majority class, position only, TF-IDF + logistic regression (sentence only / + position) |
| 8 | save `p3/data/docs.pkl`, `meta.json`, dev truth files in the official format |

**Kaggle settings**: Accelerator **None** (CPU, a few minutes), Internet **ON**, secret `HF` (optional).
**Input**: the **NB1 output** (`01_data_tokenizer_evalset`) — only for `tokenizer/bpe32768/tokenizer.json`.
**Output**: `/kaggle/working/cs781_p1/p3/data/…` → attach this notebook's output to P3_02.

In [1]:
!pip -q install tokenizers ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 3.4 MB/s eta 0:00:00


In [2]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /tmp/cs781_scratch: free 1148.8 GB / total 8758.4 GB
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
HF token found: True


## 0. Configuration and download

In [3]:
import pickle
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
try: display
except NameError: display = print

P3_DATA = ROOT / "p3" / "data"; P3_DATA.mkdir(parents=True, exist_ok=True)
P3_REPO_DIR = "Phase_3_RR"
P3_FILES = {("CL", "train"): "Train/CL_train.json", ("CL", "dev"): "Train/CL_dev.json",
            ("IT", "train"): "Train/IT_train.json", ("IT", "dev"): "Train/IT_dev.json",
            ("CL", "test"): "Test/CL_test.json", ("IT", "test"): "Test/IT_test.json"}
TOKENIZER = "bpe32768"
P3_LOCAL = os.environ.get("CS781_P3_LOCAL")          # only for local testing: a folder with Train/ and Test/

def get_p3_file(rel):
    if P3_LOCAL: return Path(P3_LOCAL) / rel
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, f"{P3_REPO_DIR}/{rel}", repo_type="dataset",
                                token=HF_TOKEN, local_dir=str(SCRATCH / "p3raw")))

# label vocabulary: same parsing rules as the official evaluator (list, label->id or id->label)
vocab_raw = load_json(get_p3_file("Train/label_vocab.json"))
if isinstance(vocab_raw, list): LABELS = list(vocab_raw)
elif all(isinstance(v, int) and not isinstance(v, bool) for v in vocab_raw.values()):
    LABELS = [k for k, _ in sorted(vocab_raw.items(), key=lambda kv: kv[1])]
else:
    LABELS = [v for _, v in sorted(vocab_raw.items(), key=lambda kv: int(kv[0]))]
assert len(LABELS) == 13 and len(set(LABELS)) == 13, LABELS
save_json(LABELS, P3_DATA / "labels.json")
print("label_vocab.json:", vocab_raw)
print("13 labels:", LABELS)

RAW = {k: load_json(get_p3_file(v)) for k, v in P3_FILES.items()}

def describe(x, depth=0, maxd=3):
    pad = "  " * depth
    if isinstance(x, dict):
        k = next(iter(x)) if x else None
        print(f"{pad}dict with {len(x)} keys; first keys {list(x)[:6]}")
        if k is not None and depth < maxd: describe(x[k], depth + 1)
    elif isinstance(x, list):
        print(f"{pad}list with {len(x)} items")
        if x and depth < maxd: describe(x[0], depth + 1)
    elif isinstance(x, str): print(f"{pad}str ({len(x)} chars): {x[:100]!r}")
    else: print(f"{pad}{type(x).__name__}: {x!r}")
for k in RAW: print("---", k); describe(RAW[k])

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

label_vocab.json: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}
13 labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

--- ('CL', 'train')
list with 40 items
  dict with 3 keys; first keys ['id', 'text', 'labels']
    str (71 chars): 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733'
--- ('CL', 'dev')
list with 5 items
  dict with 3 keys; first keys ['id', 'text', 'labels']
    str (83 chars): 'CCI_Noida_Software_Technology_Park_Ltd_vs_Star_India_PCO201808081816041524COM760948'
--- ('IT', 'train')
list with 40 items
  dict with 3 keys; first keys ['id', 'text', 'labels']
    str (18 chars): 'BHC_BomHC_1981_215'
--- ('IT', 'dev')
list with 5 items
  dict with 3 keys; first keys ['id', 'text', 'labels']
    str (12 chars): 'SC_2009_1357'
--- ('CL', 'test')
dict with 5 keys; first keys ['HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', 'COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224', 'SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC', 'CCI_Ms_Mineral_Enterprises_Limited_vs_Minis

## 1. Parse documents
Accepts a list of records `{id, text: [sentences], labels: [...]}` (the format in the task description) or a dict
`{id: record}`. If several label fields exist (IL-TUR also stores per-annotator labels), the field `labels` — the
final majority label — is used. Integer labels are mapped through the vocabulary.

In [4]:
def records(obj):
    if isinstance(obj, list): return [(r.get("id", r.get("doc_id")), r) for r in obj]
    return [(k, v if isinstance(v, dict) else {"text": v}) for k, v in obj.items()]

def to_label(x):
    if isinstance(x, int) or (isinstance(x, str) and x.isdigit()): return LABELS[int(x)]
    if isinstance(x, list): return to_label(x[0])
    return x

DOCS, problems = [], Counter()
for (dom, split), obj in RAW.items():
    for did, r in records(obj):
        sents = r.get("text", r.get("sentences"))
        assert isinstance(sents, list), f"{did}: text is not a list of sentences"
        labs = r.get("labels")
        if split != "test":
            assert labs is not None, f"{did}: no labels in a {split} file"
            labs = [to_label(x) for x in labs]
            if len(labs) != len(sents): problems["label/sentence count mismatch"] += 1
            bad = [x for x in labs if x not in LABELS]
            if bad: problems[f"unknown labels {set(bad)}"] += 1
        DOCS.append({"id": str(did), "domain": dom, "split": split, "sentences": [str(s) for s in sents],
                     "labels": labs if split != "test" else None, "extra_keys": sorted(set(r) - {"id", "text", "labels"})})
print("problems:", dict(problems) or "none")
assert not problems
print("other fields present in records:", Counter(tuple(d["extra_keys"]) for d in DOCS).most_common(3))

# ID checks — the scorer strips a trailing ".txt" and requires CL and IT test IDs to be disjoint
norm_id = lambda s: s[:-4] if s.endswith(".txt") else s
for split in ("train", "dev", "test"):
    ids = {dom: {norm_id(d["id"]) for d in DOCS if d["domain"] == dom and d["split"] == split} for dom in ("CL", "IT")}
    print(f"{split}: CL {len(ids['CL'])} docs, IT {len(ids['IT'])} docs, overlap {len(ids['CL'] & ids['IT'])}")
all_ids = [norm_id(d["id"]) for d in DOCS]
print("duplicate IDs across all files:", [k for k, v in Counter(all_ids).items() if v > 1][:10])
print("test IDs ending in .txt:", sum(d["id"].endswith(".txt") for d in DOCS if d["split"] == "test"))

problems: none
other fields present in records: [((), 100)]
train: CL 40 docs, IT 40 docs, overlap 0
dev: CL 5 docs, IT 5 docs, overlap 0
test: CL 5 docs, IT 5 docs, overlap 0
duplicate IDs across all files: []
test IDs ending in .txt: 0


## 2. Statistics: size, lengths, label distribution

In [5]:
rows = []
for d in DOCS:
    nw = [len(s.split()) for s in d["sentences"]]
    rows.append({"domain": d["domain"], "split": d["split"], "sentences": len(d["sentences"]), "words": sum(nw),
                 "empty_sents": sum(x == 0 for x in nw), "words_per_sent": np.mean(nw) if nw else 0})
df = pd.DataFrame(rows)
display(df.groupby(["domain", "split"]).agg(docs=("sentences", "size"), sentences=("sentences", "sum"),
        sents_per_doc_mean=("sentences", "mean"), sents_per_doc_max=("sentences", "max"),
        words_per_sent=("words_per_sent", "mean"), empty_sents=("empty_sents", "sum")).round(1))

lab_rows = []
for d in DOCS:
    if d["labels"] is None: continue
    for i, l in enumerate(d["labels"]):
        lab_rows.append({"domain": d["domain"], "split": d["split"], "label": l, "relpos": i / max(1, len(d["labels"]) - 1),
                         "doc": d["id"], "words": len(d["sentences"][i].split())})
dl = pd.DataFrame(lab_rows)
tab = dl.pivot_table(index="label", columns=["domain", "split"], values="relpos", aggfunc="count").reindex(LABELS).fillna(0).astype(int)
tot = dl[dl.split == "train"].label.value_counts()
tab["train_share_%"] = (100 * tot / tot.sum()).reindex(LABELS).fillna(0).round(2)
tab["docs_with_label"] = dl.groupby("label")["doc"].nunique().reindex(LABELS).fillna(0).astype(int)
print("sentences per label (train/dev, both domains)")
display(tab)
missing_dev = [l for l in LABELS if not ((dl.split == "dev") & (dl.label == l)).any()]
print("labels with NO dev sentence:", missing_dev)

docs  sentences  sents_per_doc_mean  sents_per_doc_max  words_per_sent  empty_sents
domain split                                                                                     
CL     dev       5       1396               279.2                848            33.5            0
       test      5       1339               267.8                942            37.8            0
       train    40      10593               264.8               1816            33.5            0
IT     dev       5        746               149.2                382            30.7            0
       test      5        858               171.6                420            39.3            0
       train    40       6252               156.3                442            34.2            0

sentences per label (train/dev, both domains)


domain                   CL         IT       train_share_% docs_with_label
split                   dev train  dev train                              
label                                                                     
Fact                    249  2979  195  1119         24.33              88
Issue                     3    17   12    90          0.64              43
ArgumentPetitioner      344   976   56   637          9.58              82
ArgumentRespondent      236   996   36   352          8.00              67
Statute                  58   367   62   732          6.52              57
Dissent                   0   284    0     0          1.69               2
PrecedentReliedUpon     142  1305  112   640         11.55              53
PrecedentNotReliedUpon    0   209   50   360          3.38              35
PrecedentOverruled        0     0    0     3          0.02               1
RulingByLowerCourt       56   438   46   436          5.19              51
RatioOfTheDecision      288  2856  115  1445         25.53              90
RulingByPresentCourt     18   113   31   194          1.82              90
None                      2    53   31   244          1.76              55

labels with NO dev sentence: ['Dissent', 'PrecedentOverruled']


## 3. Document structure: position, blocks, transitions
* **Relative position** of each label (0 = first sentence, 1 = last): if roles live in typical parts of a judgment,
  position is a strong feature — the model should get it.
* **Runs**: how often a sentence has the same label as the previous one, and the mean length of a run of one label.
  Long runs = block structure → a sequence model over sentences (BiLSTM / CRF) should help.

In [6]:
pos = dl[dl.split == "train"].groupby("label").relpos.describe(percentiles=[.1, .5, .9])[["count", "10%", "50%", "90%"]]
display(pos.reindex(LABELS).round(2))

same, n_pairs, runs, trans = 0, 0, defaultdict(list), Counter()
for d in DOCS:
    if d["labels"] is None or d["split"] != "train": continue
    labs = d["labels"]; run = 1
    for a, b in zip(labs, labs[1:]):
        n_pairs += 1
        if a == b: same += 1; run += 1
        else: runs[a].append(run); run = 1; trans[(a, b)] += 1
    if labs: runs[labs[-1]].append(run)
print(f"P(label == previous label) = {same / max(1, n_pairs):.3f}")
print("mean run length per label:", {l: round(float(np.mean(runs[l])), 1) for l in LABELS if runs[l]})
print("most common label changes:", [(f"{a}→{b}", n) for (a, b), n in trans.most_common(12)])

def show_doc(d, k=40):
    for i, (s, l) in enumerate(zip(d["sentences"][:k], d["labels"][:k])):
        print(f"{i:3d} {l:24s} {s[:150]}")
first_train = next(d for d in DOCS if d["split"] == "train")
print(f"\nexample document {first_train['id']} ({first_train['domain']}), first sentences:")
show_doc(first_train)

,count,10%,50%,90%
label,,,,
Fact,4098.0,0.03,0.19,0.72
Issue,107.0,0.01,0.36,0.74
ArgumentPetitioner,1613.0,0.12,0.33,0.68
ArgumentRespondent,1348.0,0.21,0.37,0.68
Statute,1099.0,0.16,0.44,0.75
Dissent,284.0,0.78,0.88,0.98
PrecedentReliedUpon,1945.0,0.38,0.69,0.95
PrecedentNotReliedUpon,569.0,0.12,0.51,0.85
PrecedentOverruled,3.0,0.57,0.61,0.62


P(label == previous label) = 0.869
mean run length per label: {'Fact': 11.4, 'Issue': 1.3, 'ArgumentPetitioner': 6.9, 'ArgumentRespondent': 10.1, 'Statute': 6.1, 'Dissent': 142.0, 'PrecedentReliedUpon': 10.3, 'PrecedentNotReliedUpon': 5.9, 'PrecedentOverruled': 1.5, 'RulingByLowerCourt': 5.8, 'RatioOfTheDecision': 8.1, 'RulingByPresentCourt': 2.6, 'None': 1.5}
most common label changes: [('PrecedentReliedUpon→RatioOfTheDecision', 123), ('RatioOfTheDecision→PrecedentReliedUpon', 115), ('Fact→ArgumentPetitioner', 94), ('RatioOfTheDecision→RulingByPresentCourt', 93), ('Statute→RatioOfTheDecision', 90), ('RatioOfTheDecision→Statute', 85), ('Fact→RatioOfTheDecision', 81), ('Fact→RulingByLowerCourt', 79), ('RulingByLowerCourt→Fact', 79), ('ArgumentPetitioner→RatioOfTheDecision', 60), ('ArgumentPetitioner→Fact', 58), ('RatioOfTheDecision→Fact', 48)]

example document CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733 (CL), first sentences:
  0 Fact                     Con

## 4. The official metric
`evaluate_phase3.py` pools the CL and IT documents into one list of sentences and computes
`sklearn.f1_score(y_true, y_pred, labels=<all 13 labels>, average="macro", zero_division=0)`.
Consequences:
* every one of the 13 labels counts 1/13, **including labels that never occur in the test gold** — such a label
  always scores F1 = 0, so the maximum reachable score is (#labels present in test) / 13;
* predicting an absent label costs nothing extra for *that* label (it is 0 anyway) but the sentence is still wrong
  for its true label;
* rare roles (Dissent, PrecedentOverruled, Issue, …) weigh as much as Fact or RatioOfTheDecision → class balancing matters.

In [7]:
def rr_scores(truth, pred):
    """identical arithmetic to evaluate_phase3.py: pooled sentences, macro over the 13 labels, zero_division=0"""
    yt, yp = [], []
    for did in truth:
        assert len(truth[did]) == len(pred[did]), f"{did}: sentence-count mismatch"
        yt += list(truth[did]); yp += list(pred[did])
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=LABELS, zero_division=0)
    return {"macro_f1": f1_score(yt, yp, labels=LABELS, average="macro", zero_division=0),
            "macro_p": precision_score(yt, yp, labels=LABELS, average="macro", zero_division=0),
            "macro_r": recall_score(yt, yp, labels=LABELS, average="macro", zero_division=0),
            "per_label": {l: {"P": p[i], "R": r[i], "F1": f[i], "support": int(s[i])} for i, l in enumerate(LABELS)},
            "n_sentences": len(yt)}

TRUTH_DEV = {d["id"]: d["labels"] for d in DOCS if d["split"] == "dev"}
present_dev = sorted({l for v in TRUTH_DEV.values() for l in v})
print(f"labels present in dev gold: {len(present_dev)}/13 -> ceiling of dev macro-F1 = {len(present_dev)/13:.3f}")
print("perfect prediction on dev:", round(rr_scores(TRUTH_DEV, TRUTH_DEV)["macro_f1"], 4))

labels present in dev gold: 11/13 -> ceiling of dev macro-F1 = 0.846
perfect prediction on dev: 0.8462


## 5. Tokenisation (Phase-1 tokenizer, same normalisation as pretraining)
Each sentence → `normalize(s, "collapse_all")` → BPE ids. No character offsets are needed in this phase.
An empty sentence keeps an empty id list (P3_02 represents it by its separator token).

In [8]:
from tokenizers import Tokenizer
TOK_PATH = find_artifact(f"tokenizer/{TOKENIZER}/tokenizer.json")
assert TOK_PATH is not None, "attach the NB1 output (tokenizer/bpe32768/tokenizer.json) as an input"
TOK = Tokenizer.from_file(str(TOK_PATH)); print("tokenizer:", TOK_PATH, "| vocab", TOK.get_vocab_size())

t0 = time.time()
for d in DOCS:
    d["norm"] = [normalize(s, "collapse_all") for s in d["sentences"]]
    encs = TOK.encode_batch(d["norm"]) if d["norm"] else []
    d["sent_ids"] = [np.asarray(e.ids, dtype=np.int32) for e in encs]
    if d["labels"] is not None: d["label_ids"] = np.asarray([LABELS.index(l) for l in d["labels"]], dtype=np.int64)
print(f"tokenised {sum(len(d['sent_ids']) for d in DOCS)} sentences in {time.time()-t0:.1f}s")
tl = np.array([len(x) for d in DOCS for x in d["sent_ids"]]); dt = np.array([sum(len(x) + 1 for x in d["sent_ids"]) for d in DOCS])
print(f"tokens/sentence: mean {tl.mean():.1f}, median {np.median(tl):.0f}, p95 {np.percentile(tl, 95):.0f}, max {tl.max()} | "
      f"sentences > 128 tokens: {(tl > 128).mean():.1%}, > 256: {(tl > 256).mean():.1%}")
print(f"tokens/document (with one separator per sentence): mean {dt.mean():.0f}, max {dt.max()} | "
      f"total {dt.sum()/1e6:.2f}M | sentences per 512-token window ≈ {512 / (tl.mean() + 1):.0f}")

tokenizer: /kaggle/input/datasets/aaravoswal/cs781-p1-nb1/cs781_p1/tokenizer/bpe32768/tokenizer.json | vocab 32768
tokenised 21184 sentences in 1.8s
tokens/sentence: mean 40.8, median 33, p95 92, max 1197 | sentences > 128 tokens: 1.9%, > 256: 0.3%
tokens/document (with one separator per sentence): mean 8846, max 73691 | total 0.88M | sentences per 512-token window ≈ 12


## 6. Baselines on dev (official metric)
* **majority**: always the most frequent training label.
* **position only**: the document is cut into 20 equal parts; each part predicts its most frequent training label
  (per domain). Shows how much the *position* alone explains.
* **TF-IDF + logistic regression** (classical, no neural network, class-balanced): the sentence's words only, and
  the words + position bin + domain. A lexical floor that our fine-tuned models must beat.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix

train_docs = [d for d in DOCS if d["split"] == "train"]; dev_docs = [d for d in DOCS if d["split"] == "dev"]
NBIN = 20
def pbin(i, n): return min(NBIN - 1, int(NBIN * i / max(1, n)))
base = {}
maj = Counter(l for d in train_docs for l in d["labels"]).most_common(1)[0][0]
base["majority"] = rr_scores(TRUTH_DEV, {d["id"]: [maj] * len(d["sentences"]) for d in dev_docs})

pos_tab = defaultdict(Counter)
for d in train_docs:
    for i, l in enumerate(d["labels"]): pos_tab[(d["domain"], pbin(i, len(d["labels"])))][l] += 1
pos_pred = {d["id"]: [(pos_tab[(d["domain"], pbin(i, len(d["sentences"])))] or Counter([maj])).most_common(1)[0][0]
                      for i in range(len(d["sentences"]))] for d in dev_docs}
base["position_only"] = rr_scores(TRUTH_DEV, pos_pred)

def flat(docs):
    X, P, y, owner = [], [], [], []
    for d in docs:
        n = len(d["sentences"])
        for i, s in enumerate(d["norm"]):
            X.append(s.lower()); P.append((pbin(i, n), d["domain"])); owner.append(d["id"])
            if d["labels"] is not None: y.append(d["labels"][i])
    return X, P, y, owner
Xtr, Ptr, ytr, _ = flat(train_docs); Xdv, Pdv, _, odv = flat(dev_docs)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
Atr, Adv = vec.fit_transform(Xtr), vec.transform(Xdv)
def posfeat(P):
    m = np.zeros((len(P), NBIN + 2), np.float32)
    for k, (b, dom) in enumerate(P): m[k, b] = 1; m[k, NBIN + (dom == "IT")] = 1
    return csr_matrix(m)
preds = {"position_only": pos_pred}
def unflat(pred, owner):
    out = defaultdict(list)
    for o, p in zip(owner, pred): out[o].append(p)
    return dict(out)
for name, (A, B) in {"tfidf_logreg_sentence": (Atr, Adv),
                     "tfidf_logreg_+position": (hstack([Atr, posfeat(Ptr) * 3]).tocsr(), hstack([Adv, posfeat(Pdv) * 3]).tocsr())}.items():
    clf = LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")
    clf.fit(A, ytr)
    preds[name] = unflat(clf.predict(B), odv)
    base[name] = rr_scores(TRUTH_DEV, preds[name])

btab = pd.DataFrame({k: {"macro_F1": v["macro_f1"], "macro_P": v["macro_p"], "macro_R": v["macro_r"]} for k, v in base.items()}).T
display(btab.round(4))
print("\nper-label dev F1:")
display(pd.DataFrame({k: {l: v["per_label"][l]["F1"] for l in LABELS} for k, v in base.items()}).round(3))
for dom in ("CL", "IT"):
    tr = {d["id"]: d["labels"] for d in dev_docs if d["domain"] == dom}
    print(f"{dom} dev only: " + " | ".join(f"{n} {rr_scores(tr, {k: p[k] for k in tr})['macro_f1']:.4f}" for n, p in preds.items()))

,macro_F1,macro_P,macro_R
majority,0.0244,0.0145,0.0769
position_only,0.1427,0.1763,0.1636
tfidf_logreg_sentence,0.2944,0.2982,0.3026
tfidf_logreg_+position,0.3520,0.3600,0.3530



per-label dev F1:


,majority,position_only,tfidf_logreg_sentence,tfidf_logreg_+position
Fact,0.000,0.562,0.549,0.615
Issue,0.000,0.000,0.353,0.444
ArgumentPetitioner,0.000,0.205,0.319,0.384
ArgumentRespondent,0.000,0.126,0.268,0.402
Statute,0.000,0.000,0.643,0.658
Dissent,0.000,0.000,0.000,0.000
PrecedentReliedUpon,0.000,0.161,0.323,0.381
PrecedentNotReliedUpon,0.000,0.000,0.179,0.207
PrecedentOverruled,0.000,0.000,0.000,0.000
RulingByLowerCourt,0.000,0.000,0.188,0.153


CL dev only: position_only 0.1252 | tfidf_logreg_sentence 0.2321 | tfidf_logreg_+position 0.2993
IT dev only: position_only 0.1109 | tfidf_logreg_sentence 0.3313 | tfidf_logreg_+position 0.3334


## 7. Save for P3_02

In [10]:
keep = ["id", "domain", "split", "sentences", "labels", "norm", "sent_ids", "label_ids"]
with open(P3_DATA / "docs.pkl.tmp", "wb") as f:
    pickle.dump([{k: d.get(k) for k in keep} for d in DOCS], f, protocol=4)
os.replace(P3_DATA / "docs.pkl.tmp", P3_DATA / "docs.pkl")
for dom in ("CL", "IT"):      # dev truth in the official format (one file per domain, like the scorer)
    save_json({d["id"]: d["labels"] for d in dev_docs if d["domain"] == dom}, P3_DATA / f"truth_dev_{dom}.json")
meta = {"labels": LABELS, "tokenizer": TOKENIZER,
        "counts": {f"{dom}_{sp}": sum(1 for d in DOCS if d["domain"] == dom and d["split"] == sp) for dom in ("CL", "IT") for sp in ("train", "dev", "test")},
        "train_label_counts": {l: int(tot.get(l, 0)) for l in LABELS},
        "labels_missing_in_dev": missing_dev,
        "p_same_as_previous": same / max(1, n_pairs),
        "tokens_per_sentence": {"mean": float(tl.mean()), "p95": float(np.percentile(tl, 95)), "max": int(tl.max())},
        "tokens_per_doc": {"mean": float(dt.mean()), "max": int(dt.max())},
        "baselines_dev_macro_f1": {k: v["macro_f1"] for k, v in base.items()}}
save_json(meta, P3_DATA / "meta.json")
print(json.dumps({k: v for k, v in meta.items() if k != "labels"}, indent=1))
print("saved:", sorted(p.name for p in P3_DATA.iterdir()))

{
 "tokenizer": "bpe32768",
 "counts": {
  "CL_train": 40,
  "CL_dev": 5,
  "CL_test": 5,
  "IT_train": 40,
  "IT_dev": 5,
  "IT_test": 5
 },
 "train_label_counts": {
  "Fact": 4098,
  "Issue": 107,
  "ArgumentPetitioner": 1613,
  "ArgumentRespondent": 1348,
  "Statute": 1099,
  "Dissent": 284,
  "PrecedentReliedUpon": 1945,
  "PrecedentNotReliedUpon": 569,
  "PrecedentOverruled": 3,
  "RulingByLowerCourt": 874,
  "RatioOfTheDecision": 4301,
  "RulingByPresentCourt": 307,
  "None": 297
 },
 "labels_missing_in_dev": [
  "Dissent",
  "PrecedentOverruled"
 ],
 "p_same_as_previous": 0.8689531762600656,
 "tokens_per_sentence": {
  "mean": 40.75967711480362,
  "p95": 92.0,
  "max": 1197
 },
 "tokens_per_doc": {
  "mean": 8846.37,
  "max": 73691
 },
 "baselines_dev_macro_f1": {
  "majority": 0.0243614931237721,
  "position_only": 0.14269967812906964,
  "tfidf_logreg_sentence": 0.2943704115227044,
  "tfidf_logreg_+position": 0.3520123077557354
 }
}
saved: ['docs.pkl', 'labels.json', 'meta.json